# 02 — Baseline e Thompson Sampling 

## Objetivo
1. Construir uma baseline com uma regra fixa
2. Construir o algoritmo thompson Sampling 
3. Comparar ambas as arbordagens

- **Braços:** `cellular` e `telephone`
- **Recompensa:** `y = yes` → 1; `y = no` → 0
- **Contexto:** variáveis disponíveis antes do contato

## Configurando MLRuns

In [30]:
import mlflow

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("bank_marketing_bandit")

print(mlflow.get_tracking_uri())

sqlite:///../mlflow.db


## 1. Imports


In [31]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

np.random.seed(42)
pd.set_option("display.max_columns", None)

## 2. Carregamento dos dados


In [32]:
df = pd.read_csv("../data/raw/bank-additional-full.csv", sep=";")

df["reward"] = (df["y"] == "yes").astype(int)
df = df.drop_duplicates()
df = df.drop(columns=["duration"])

bins = [0, 25, 35, 45, 55, 65, 75, 100]
labels = ["Até 25", "26-35", "36-45", "46-55", "56-65", "66-75", "76+"]

df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels)

print(f"Linhas: {len(df):,}")
print(f"Colunas: {df.shape[1]}")
print(f"Braços: {sorted(df['contact'].unique())}")
print(f"Taxa global de recompensa: {df['reward'].mean():.2%}")

Linhas: 41,176
Colunas: 22
Braços: ['cellular', 'telephone']
Taxa global de recompensa: 11.27%


## 3. Separando bases

In [33]:
train, test = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["reward"],
)

## 4. Baseline

### Regra fixa: sempre escolher o braço com melhor histórico

In [34]:
arm_conversion = train.groupby("contact")["reward"].mean()
print(arm_conversion)

contact
cellular     0.147476
telephone    0.051912
Name: reward, dtype: float64


In [35]:
best_arm_baseline = arm_conversion.idxmax()
print(f"\nMelhor braço pelo histórico de treino: {best_arm_baseline}")


Melhor braço pelo histórico de treino: cellular


In [36]:
# 3. Filtra o teste para avaliar apenas o braço escolhido (Replay Method)
baseline_test = test[test["contact"] == best_arm_baseline].copy()
baseline_conversion = baseline_test["reward"].mean()

print(f"Baseline: sempre escolher '{best_arm_baseline}'")
print(f"Taxa de conversão no teste: {baseline_conversion:.2%}")
print(f"Total de observações válidas no teste: {len(baseline_test)} de {len(test)}")

Baseline: sempre escolher 'cellular'
Taxa de conversão no teste: 14.70%
Total de observações válidas no teste: 5196 de 8236


#### MLFlow

In [37]:
with mlflow.start_run(run_name="baseline"):

    mlflow.log_param("algorithm", "Baseline")
    mlflow.log_param("strategy", f"always_{best_arm_baseline}")

    mlflow.log_metric("conversion", baseline_conversion)
    mlflow.log_metric("evaluated_cases", len(baseline_test))
    mlflow.log_metric(
        "feedback_rate",
        len(baseline_test) / len(test),
    )

## 5. Thompson Sampling

In [38]:
arms = sorted(train["contact"].unique())

alpha = train.groupby("contact")["reward"].sum().to_dict()
beta = (
    train.groupby("contact")["reward"].count()
    - train.groupby("contact")["reward"].sum()
).to_dict()

### Tradicional

Aqui basicamente o thompson sampling escolhe o braço com o maior histórico de reward, o resultado final se assemelha muito ao baseline que foi criado

In [39]:
# Garantir a priori Beta(1,1) para braços eventualmente sem dados
for arm in arms:
    alpha.setdefault(arm, 1)
    beta.setdefault(arm, 1)

ts_test_results = []

In [40]:
for _, row in test.iterrows():
    # Amostra uma taxa da distribuição Beta para cada braço
    samples = {arm: np.random.beta(alpha[arm], beta[arm]) for arm in arms}

    # Escolhe o braço com maior amostra
    selected_arm = max(samples, key=samples.get)

    observed_arm = row["contact"]
    observed_reward = row["reward"]

    # Avaliação via Replay Method
    if selected_arm == observed_arm:
        reward = observed_reward

        # 2. ATUALIZAÇÃO DOS PARÂMETROS
        alpha[selected_arm] += reward
        beta[selected_arm] += 1 - reward
    else:
        reward = np.nan

    ts_test_results.append(
        {
            "selected_arm": selected_arm,
            "observed_arm": observed_arm,
            "reward": reward,
        }
    )

ts_test_results = pd.DataFrame(ts_test_results)

In [41]:
evaluated = ts_test_results.dropna(subset=["reward"])

ts_conversion = evaluated["reward"].mean()

print(f"Casos avaliados: {len(evaluated)}")
print(f"Taxa de feedback: {len(evaluated) / len(test):.2%}")
print(f"Conversão Thompson (casos avaliados): {ts_conversion:.2%}")

Casos avaliados: 5196
Taxa de feedback: 63.09%
Conversão Thompson (casos avaliados): 14.70%


In [42]:
with mlflow.start_run(run_name="thompson_sampling"):

    mlflow.log_param("algorithm", "Thompson Sampling")
    mlflow.log_param("prior", "Beta(1,1)")
    mlflow.log_param("arms", ", ".join(arms))

    mlflow.log_metric("conversion", ts_conversion)
    mlflow.log_metric("feedback_rate", len(evaluated) / len(test))
    mlflow.log_metric("evaluated_cases", len(evaluated))

### Contextual

Aqui observamos o contexto do cliente para definir qual braço iremos escolher

In [43]:
context_features = [
    "age_group",
    "poutcome",
    "campaign",
    "month"
]

categorical_features = [
    "age_group",
    "poutcome",
    "campaign",
    "month"
]
 
numeric_features = [
]

#### Treinando modelo cellular

In [44]:
cellular_train = train[train["contact"] == "cellular"].copy()
X_cellular = cellular_train[context_features]
y_cellular = cellular_train["reward"]

cellular_model = Pipeline(
    [
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    (
                        "categorical",
                        OneHotEncoder(handle_unknown="ignore"),
                        categorical_features,
                    ),
                    (
                        "numeric",
                        StandardScaler(),
                        numeric_features,
                    ),
                ]
            ),
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

In [45]:
cellular_model.fit(X_cellular, y_cellular)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['age_group','poutcome','campaign','month']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns 

#### Treinando modelo telephone

In [46]:
telephone_train = train[train["contact"] == "telephone"].copy()

X_telephone = telephone_train[context_features]
y_telephone = telephone_train["reward"]

telephone_model = Pipeline(
    [
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    (
                        "categorical",
                        OneHotEncoder(handle_unknown="ignore"),
                        categorical_features,
                    ),
                    (
                        "numeric",
                        StandardScaler(),
                        numeric_features,
                    ),
                ]
            ),
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

In [47]:
telephone_model.fit(X_telephone, y_telephone)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['age_group','poutcome','campaign','month']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns 

#### Testando (Com Amostragem Thompson + Avaliação Replay)

In [48]:
X_test_context = test[context_features]

In [49]:
# 1. Probabilidades base estimadas pelos modelos
p_cellular = cellular_model.predict_proba(X_test_context)[:, 1]
p_telephone = telephone_model.predict_proba(X_test_context)[:, 1]

# 2. Injeção de incerteza (Thompson Sampling)
# Amostramos da distribuição da probabilidade prevista considerando o erro de estimativa
np.random.seed(42)
std_dev = 0.05  # Desvio padrão que representa a incerteza do modelo

sampled_p_cellular = np.random.normal(p_cellular, scale=std_dev)
sampled_p_telephone = np.random.normal(p_telephone, scale=std_dev)

# Garantir que as amostras fiquem no intervalo [0, 1]
sampled_p_cellular = np.clip(sampled_p_cellular, 0, 1)
sampled_p_telephone = np.clip(sampled_p_telephone, 0, 1)

In [50]:
# 3. Decisão Estocástica do Thompson Sampling
test_eval = test.copy()
test_eval["selected_arm"] = np.where(
    sampled_p_cellular >= sampled_p_telephone, "cellular", "telephone"
)

# 4. Avaliação Offline via Replay Method (Mesma regra do Baseline e do TS Tradicional)
test_eval["evaluated_reward"] = np.where(
    test_eval["selected_arm"] == test_eval["contact"], test_eval["reward"], np.nan
)

In [51]:
evaluated_cts = test_eval.dropna(subset=["evaluated_reward"])
cts_conversion = evaluated_cts["evaluated_reward"].mean()

print(f"Casos avaliados no Contextual: {len(evaluated_cts)}")
print(f"Taxa de feedback: {len(evaluated_cts) / len(test):.2%}")
print(f"Conversão Contextual TS (Replay): {cts_conversion:.2%}")

Casos avaliados no Contextual: 3544
Taxa de feedback: 43.03%
Conversão Contextual TS (Replay): 15.24%


#### MlFlow

In [52]:
with mlflow.start_run(run_name="contextual_thompson_sampling"):

    mlflow.log_param("algorithm", "Contextual Thompson Sampling")
    mlflow.log_param("context_features", ", ".join(context_features))
    mlflow.log_param("categorical_features", ", ".join(categorical_features))
    mlflow.log_param("numeric_features", ", ".join(numeric_features))
    mlflow.log_param("std_dev", 0.05)

    mlflow.log_metric("conversion", cts_conversion)
    mlflow.log_metric(
        "feedback_rate",
        len(evaluated_cts) / len(test),
    )
    mlflow.log_metric(
        "evaluated_cases",
        len(evaluated_cts),
    )

### Bootstrap Thompson Sampling

O bootstrap também utiliza do contexto do cliente, porém estila o modelo inumeros vezes assim verificando também a incerteza das estimativas e utilizando durante a tomada de decisão.

In [53]:
### Bootstrap Thompson Sampling
N_BOOTSTRAPS = 100

bootstrap_models = {arm: [] for arm in arms}

# 1. Treinamento dos Modelos Bootstrap
for arm in arms:
    arm_data = train[train["contact"] == arm].copy()

    for i in range(N_BOOTSTRAPS):
        bootstrap_sample = arm_data.sample(
            n=len(arm_data),
            replace=True,
            random_state=42 + i,
        )

        X_bootstrap = bootstrap_sample[context_features]
        y_bootstrap = bootstrap_sample["reward"]

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "categorical",
                    OneHotEncoder(handle_unknown="ignore"),
                    categorical_features,
                ),
                (
                    "numeric",
                    StandardScaler(),
                    numeric_features,
                ),
            ]
        )

        model = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                (
                    "model",
                    LogisticRegression(
                        max_iter=1000,
                        random_state=42,
                    ),
                ),
            ]
        )

        model.fit(X_bootstrap, y_bootstrap)
        bootstrap_models[arm].append(model)

In [54]:
# 2. Loop de Decisão Estocástica
rng = np.random.default_rng(42)
ts_decisions = []
diffs = []

for _, row in test.iterrows():
    customer = row[context_features].to_frame().T
    sampled_predictions = {}

    for arm in arms:
        models = bootstrap_models[arm]

        # Sorteia aleatoriamente um dos N modelos bootstrap do braço
        sampled_model = models[rng.integers(len(models))]

        # Estima a probabilidade de reward usando o modelo sorteado
        sampled_probability = sampled_model.predict_proba(customer)[0, 1]
        sampled_predictions[arm] = sampled_probability

    # Mede a diferença entre as probabilidades dos braços
    diff = abs(
        sampled_predictions["cellular"]
        - sampled_predictions["telephone"]
    )
    diffs.append(diff)

    # Escolhe o braço com maior recompensa estimada
    selected_arm = max(
        sampled_predictions,
        key=sampled_predictions.get,
    )
    ts_decisions.append(selected_arm)

print("Diferença média:", np.mean(diffs))
print("Diferença mediana:", np.median(diffs))

Diferença média: 0.08057428676799934
Diferença mediana: 0.052663748964528315


In [55]:
# 3. Avaliação de Desempenho via Replay Method
test_bootstrap = test.copy()
test_bootstrap["selected_arm"] = ts_decisions

test_bootstrap["evaluated_reward"] = np.where(
    test_bootstrap["selected_arm"] == test_bootstrap["contact"],
    test_bootstrap["reward"],
    np.nan,
)

evaluated_bts = test_bootstrap.dropna(subset=["evaluated_reward"])
bts_conversion = evaluated_bts["evaluated_reward"].mean()

print(f"Casos avaliados no Bootstrap TS: {len(evaluated_bts)}")
print(f"Taxa de feedback: {len(evaluated_bts) / len(test):.2%}")
print(f"Conversão Bootstrap TS (Replay): {bts_conversion:.2%}")

# 4. Comparativo de Proporção de Seleção dos Braços
comparison = pd.DataFrame(
    {"Bootstrap TS": test_bootstrap["selected_arm"].value_counts(normalize=True)}
).fillna(0)

print("\n--- Distribuição de Seleção dos Braços ---")
print(comparison)

Casos avaliados no Bootstrap TS: 3478
Taxa de feedback: 42.23%
Conversão Bootstrap TS (Replay): 15.07%

--- Distribuição de Seleção dos Braços ---
              Bootstrap TS
selected_arm              
cellular          0.748179
telephone         0.251821


#### MlFlow

In [56]:
with mlflow.start_run(run_name="bootstrap_thompson_sampling"):

    mlflow.log_param(
        "algorithm",
        "Bootstrap Thompson Sampling",
    )

    mlflow.log_param(
        "n_bootstraps",
        N_BOOTSTRAPS,
    )

    mlflow.log_param(
        "context_features",
        ", ".join(context_features),
    )

    mlflow.log_param(
        "categorical_features",
        ", ".join(categorical_features),
    )

    mlflow.log_param(
        "numeric_features",
        ", ".join(numeric_features),
    )

    mlflow.log_metric(
        "conversion",
        bts_conversion,
    )

    mlflow.log_metric(
        "feedback_rate",
        len(evaluated_bts) / len(test),
    )

    mlflow.log_metric(
        "evaluated_cases",
        len(evaluated_bts),
    )

    mlflow.log_metric(
        "mean_diff",
        np.mean(diffs),
    )

    mlflow.log_metric(
        "median_diff",
        np.median(diffs),
    )

    mlflow.log_metric(
        "cellular_pct",
        test_bootstrap["selected_arm"]
        .value_counts(normalize=True)
        .get("cellular", 0),
    )

    mlflow.log_metric(
        "telephone_pct",
        test_bootstrap["selected_arm"]
        .value_counts(normalize=True)
        .get("telephone", 0),
    )

### Comparando

In [57]:
# Tabela Consolidada com as 4 Abordagens
summary_metrics = pd.DataFrame(
    {
        "Estratégia": [
            "Baseline (Fixed Best Arm)",
            "Thompson Sampling Tradicional",
            "Contextual TS (Normal Approximation)",
            "Bootstrap Thompson Sampling",
        ],
        "Taxa de Conversão Observada": [
            baseline_conversion,
            ts_conversion,
            cts_conversion,
            bts_conversion,
        ],
    }
)

summary_metrics["Taxa de Conversão Observada"] = summary_metrics[
    "Taxa de Conversão Observada"
].apply(lambda x: f"{x:.2%}")
print(summary_metrics.to_string(index=False))

                          Estratégia Taxa de Conversão Observada
           Baseline (Fixed Best Arm)                      14.70%
       Thompson Sampling Tradicional                      14.70%
Contextual TS (Normal Approximation)                      15.24%
         Bootstrap Thompson Sampling                      15.07%


## 6. Simulação

In [58]:
golden_set = test.sample(
    n=1000,
    random_state=42,
).copy()
rng_golden = np.random.default_rng(123)
golden_results = []

for idx, row in golden_set.iterrows():
    customer = row[context_features].to_frame().T
    sampled_predictions = {}

    for arm in arms:
        models = bootstrap_models[arm]

        # Sorteia um modelo bootstrap para representar a incerteza associada ao braço
        sampled_model = models[rng_golden.integers(len(models))]

        probability = sampled_model.predict_proba(customer)[0, 1]
        sampled_predictions[arm] = probability

    selected_arm = max(
        sampled_predictions,
        key=sampled_predictions.get,
    )

    golden_results.append(
        {
            "customer_id": idx,
            "age": row["age"],
            "job": row["job"],
            "poutcome": row["poutcome"],
            "contact_real": row["contact"],
            "reward_real": row["reward"],
            "prob_cellular": sampled_predictions["cellular"],
            "prob_telephone": sampled_predictions["telephone"],
            "recommended_arm": selected_arm,
            "decision_matches_real": selected_arm == row["contact"],
        }
    )

golden_df = pd.DataFrame(golden_results)

golden_df

,customer_id,age,job,poutcome,contact_real,reward_real,prob_cellular,prob_telephone,recommended_arm,decision_matches_real
0,6409,51,retired,nonexistent,telephone,0,0.092151,0.029062,cellular,False
1,569,43,unemployed,nonexistent,telephone,0,0.076453,0.034055,cellular,False
2,18430,49,technician,nonexistent,cellular,0,0.090724,0.055024,cellular,True
3,1230,48,blue-collar,nonexistent,telephone,0,0.046168,0.039269,cellular,False
4,2454,54,blue-collar,nonexistent,telephone,0,0.095195,0.026510,cellular,False
...,...,...,...,...,...,...,...,...,...,...
995,22462,30,admin.,nonexistent,cellular,0,0.098879,0.116499,telephone,False
996,19441,35,technician,nonexistent,cellular,0,0.104517,0.141257,telephone,False
997,12439,41,blue-collar,nonexistent,cellular,0,0.079726,0.065361,cellular,True
998,15478,24,blue-collar,nonexistent,cellular,1,0.125589,0.018287,cellular,True
